# Liên hệ Mail p.baoton.l@gmail.com để được cung cấp tài khoản login 

In [33]:
tai_khoan = ""
mat_khau = ""

In [34]:
# Import cac thu vien can thiet
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import TimeoutException

import time
import pandas as pd

# **Thu thập danh sách học phần có trong chương tình đào tạo DBS K39**

In [35]:
def lay_du_lieu_mon_hoc(tai_khoan, mat_khau):
    """
    Hàm tự động đăng nhập vào trang online.hub.edu.vn và cào dữ liệu môn học.
    Trả về: Pandas DataFrame chứa danh sách các môn học.
    """
    # Khởi tạo ban đầu
    chrome_options = Options()
    # chrome_options.add_argument('--kiosk-printing')
    # chrome_options.add_argument("--headless") # Mở comment dòng này nếu muốn chạy ngầm (không hiện trình duyệt)
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    wait = WebDriverWait(driver, 10)
    
    # Khởi tạo mảng rỗng để lưu trữ dữ liệu (đặt ở ngoài try để đảm bảo luôn tồn tại khi return)
    danh_sach_hoc_phan = []
    
    try:
        # Mở web
        driver.get("https://online.hub.edu.vn/")
        
        # Click nút Đăng Nhập
        login_button_first_interface = driver.find_element(By.CSS_SELECTOR, "#lbtDangnhap.menutop[href*='__doPostBack']")
        login_button_first_interface.click()

        # Đăng nhập Username, Password
        username_placeholder = wait.until(EC.presence_of_element_located((By.ID, "ContentPlaceHolder1_ctl00_ctl00_txtUserName")))
        password_placeholder = driver.find_element(By.ID, "ContentPlaceHolder1_ctl00_ctl00_txtPassword")
        
        username_placeholder.clear()
        password_placeholder.clear()
        
        # Điền thông tin đăng nhập từ tham số truyền vào hàm
        username_placeholder.send_keys(tai_khoan)
        password_placeholder.send_keys(mat_khau)

        # Click nút Đăng Nhập
        login_button_login_interface = driver.find_element(By.ID, "ContentPlaceHolder1_ctl00_ctl00_btLogin")
        login_button_login_interface.click()

        # Click vào mục Chương trình đào tạo
        study_program = wait.until(EC.element_to_be_clickable((By.ID, "ContentPlaceHolder1_ctl00_ctl00_lnkStudyProgram")))
        study_program.click()
        
        # Tìm tất cả các dòng <tr> chứa dữ liệu
        danh_sach_dong = wait.until(EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "#ContentPlaceHolder1_ctl00_ctl00_ctl00_tbSource tbody tr")
        ))
        
        # Lặp qua từng dòng để bóc tách dữ liệu
        for dong in danh_sach_dong:
            cac_cot = dong.find_elements(By.TAG_NAME, "td")
            
            if len(cac_cot) >= 4:
                ten_hp = cac_cot[2].text.strip()  
                loai_hp = cac_cot[3].text.strip() 
                
                danh_sach_hoc_phan.append({
                    "Tên học phần": ten_hp,
                    "Loại học phần": loai_hp
                })
                
        # Loại bỏ 3 phần tử đầu tiên (thường là header/tiêu đề bị thừa như trong code cũ của bạn)
        if len(danh_sach_hoc_phan) > 3:
            danh_sach_hoc_phan = danh_sach_hoc_phan[3:]

    except Exception as e:
        print(f"Quá trình cào dữ liệu gặp lỗi: {e}")
    finally:
        # Luôn đảm bảo đóng trình duyệt sau khi chạy xong để giải phóng RAM
        driver.quit()

    # Chuyển đổi list dictionary thành DataFrame và trả về kết quả
    df_monhoc = pd.DataFrame(danh_sach_hoc_phan)
    return df_monhoc

# **Gọi hàm thu thập dữ liệu môn học**

In [36]:
df_monhoc = lay_du_lieu_mon_hoc(tai_khoan, mat_khau)

display(df_monhoc)

,Tên học phần,Loại học phần
0,An toàn bảo mật thông tin trong kinh doanh,Tự Chọn
1,Chủ nghĩa xã hội khoa học,Bắt Buộc
2,Chuẩn công nghệ thông tin đầu vào,Bắt Buộc
3,Chuẩn ngoại ngữ đầu vào,Bắt Buộc
4,Chuỗi khối,Bắt Buộc
...,...,...
58,Toán cao cấp 2,Bắt Buộc
59,Trí tuệ nhân tạo,Bắt Buộc
60,Triết học Mác - Lênin,Bắt Buộc
61,Trực quan hóa dữ liệu,Tự Chọn


# **Thu thập dữ liệu điểm số sinh viên DBS K39**

In [30]:
def lay_diem_tong_hop(tai_khoan, mat_khau):
    """
    Hàm đăng nhập và cào điểm của một danh sách sinh viên.
    Trả về: DataFrame chứa điểm tổng hợp của tất cả sinh viên trong mssv_list.
    """
    # Bắt đầu tính thời gian
    start_time = time.time()

    # Khởi tạo ban đầu
    chrome_options = Options()
    # chrome_options.add_argument("--headless") # Bỏ comment nếu muốn chạy ngầm
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    
    wait = WebDriverWait(driver, 10)
    wait1 = WebDriverWait(driver, 1.5)
    
    danh_sach_tong_hop = []
    df_tong_hop = pd.DataFrame() # Khởi tạo sẵn phòng trường hợp lỗi vẫn có biến để return

    root_id = "03023923"
    mssv_list = []
    for i in range(1, 306):
        mssv_moi = f"{root_id}{i:04d}"
        mssv_list.append(mssv_moi)

    try:
        # Mở web
        driver.get("https://online.hub.edu.vn/")
        
        # Click nút Đăng Nhập
        login_button_first_interface = driver.find_element(By.CSS_SELECTOR, "#lbtDangnhap.menutop[href*='__doPostBack']")
        login_button_first_interface.click()

        # Đăng nhập Username, Password
        username_placeholder = wait.until(EC.presence_of_element_located((By.ID, "ContentPlaceHolder1_ctl00_ctl00_txtUserName")))
        password_placeholder = driver.find_element(By.ID, "ContentPlaceHolder1_ctl00_ctl00_txtPassword")
        
        username_placeholder.clear()
        password_placeholder.clear()
        username_placeholder.send_keys(tai_khoan)
        password_placeholder.send_keys(mat_khau)

        # Click nút Đăng Nhập
        login_button_login_interface = driver.find_element(By.ID, "ContentPlaceHolder1_ctl00_ctl00_btLogin")
        login_button_login_interface.click()

        # Tắt hộp thoại in (print dialog) tự động bật lên
        driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
            "source": "window.print = function() {};"
        })
        
        # Lấy danh sách môn học chuẩn từ df_monhoc
        danh_sach_mon_chuan = df_monhoc['Tên học phần'].tolist()

        # BẮT ĐẦU VÒNG LẶP DUYỆT TỪNG SINH VIÊN
        for mssv in mssv_list: 
            url = f"https://online.hub.edu.vn/Portlets/uis_Myspace/Student/Graduations/GraduationMarks.aspx?Studentid={mssv}&StudyProgramID=K39DH5234040505"
            driver.get(url)
            
            try:
                rows = wait1.until(EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "#grvRegisters tbody tr")
                ))
            except TimeoutException:
                continue # Bỏ qua nếu không tìm thấy bảng điểm
        
            # Lấy thông tin cá nhân
            hoten = driver.find_element(By.ID , "lblFullName").text
            
            # Tạo gói dữ liệu ĐỘC LẬP cho sinh viên này
            du_lieu_sinh_vien = {
                "MSSV": mssv,
                "Họ Tên": hoten
            }
            
            # Trải sẵn các cột môn học với giá trị rỗng
            for mon in danh_sach_mon_chuan:
                du_lieu_sinh_vien[mon] = None
            
            # Cào và điền điểm
            for row in rows[1:]:
                categories = row.find_elements(By.TAG_NAME, "td")
                if len(categories) >= 6:
                    ten_hoc_phan = categories[2].text.strip()
                    diem_10 = categories[4].text.strip()
            
                    if ten_hoc_phan in du_lieu_sinh_vien:
                        du_lieu_sinh_vien[ten_hoc_phan] = diem_10
        
            # Ném Dictionary của người đó vào thùng chứa tổng
            danh_sach_tong_hop.append(du_lieu_sinh_vien)

        # Tạo DataFrame và ép buộc thứ tự cột
        if danh_sach_tong_hop:
            df_tong_hop = pd.DataFrame(danh_sach_tong_hop)
            thu_tu_cot = ["MSSV", "Họ Tên"] + danh_sach_mon_chuan
            df_tong_hop = df_tong_hop.reindex(columns=thu_tu_cot)

    except Exception as e:
        print(f"Quá trình chạy gặp lỗi: {e}")
        
    finally:
        # Luôn đảm bảo trình duyệt được đóng
        driver.quit()

    # Kết thúc bộ đếm và in ra dòng duy nhất
    end_time = time.time()
    execution_time = round(end_time - start_time, 2)
    print(f"Done. Thời gian thực hiện: {execution_time} giây")
    
    # Trả về kết quả
    return df_tong_hop

# **Gọi hàm thu thập dữ liệu điểm**

In [31]:
df_score  = lay_diem_tong_hop(tai_khoan, mat_khau)
display(df_score)

Done. Thời gian thực hiện: 589.45 giây


,MSSV,Họ Tên,An toàn bảo mật thông tin trong kinh doanh,Chủ nghĩa xã hội khoa học,Chuẩn công nghệ thông tin đầu vào,Chuẩn ngoại ngữ đầu vào,Chuỗi khối,Cơ sở dữ liệu,Đạo đức và văn hóa doanh nghiệp,Giải thuật ứng dụng trong kinh doanh,...,Thương mại điện tử,Tiếng Anh chuyên ngành 1,Tiếng anh chuyên ngành Khoa học dữ liệu trong kinh doanh,Tin học ứng dụng,Toán cao cấp 1,Toán cao cấp 2,Trí tuệ nhân tạo,Triết học Mác - Lênin,Trực quan hóa dữ liệu,Tư tưởng Hồ Chí Minh
0,030239230003,Nguyễn Thị Thùy An,None,None,R,R,8.0,6.7,None,7.6,...,None,None,None,6.5,6.0,7.2,None,6.9,None,8.5
1,030239230005,Đào Việt Anh,None,None,R,R,8.5,7.6,None,8.8,...,None,None,None,7.5,7.6,9.5,None,7.4,None,9.0
2,030239230008,Nguyễn Hoàng Trúc Anh,None,None,None,None,None,None,None,None,...,None,None,None,None,5.8,4.8,None,7.6,None,None
3,030239230009,Nguyễn Hồng Anh,None,None,R,R,8.1,7.4,None,7.0,...,None,None,None,7.4,8.7,7.6,None,7.2,None,8.4
4,030239230020,Tôn Thất Gia Bảo,None,None,R,R,8.1,8.3,None,9.1,...,8.3,7.9,None,10.0,8.9,9.7,None,7.3,None,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,030239230293,Tào Quang Vũ,None,8.0,R,R,7.6,7.1,None,8.1,...,None,7.8,None,8.7,7.3,5.6,None,7.4,None,8.8
84,030239230294,Nguyễn Thị Thảo Vy,None,None,None,R,None,VT,None,None,...,None,None,None,None,7.0,6.0,None,6.8,None,None
85,030239230296,Trần Thị Ngọc Vy,None,None,R,R,7.5,7.7,None,6.5,...,None,8.0,None,None,5.3,5.8,None,7.5,None,7.7
86,030239230297,Trần Thúy Vy,None,None,R,R,7.3,7.4,None,7.7,...,None,7.5,None,9.1,6.9,6.7,None,6.9,None,7.6


# **Lưu dữ liệu dạng excel (.xlsx)**

In [37]:
#df_monhoc.to_excel("monhoc.xlsx", index = False)
# df_score.to_excel("khdl_score_k39.xlsx", index = False)